# Predict account closure with direct KumoTabular

Which accounts are likely to close voluntarily in the next 30 days? Use synthetic account aggregates with pretrained KumoTabular to evaluate later held-out accounts, score a prospective cohort, and inspect sensitivity for two accounts.

Run locally on Python 3.12 or newer; this example uses the small model on CPU. The first run downloads public [Kumo-Tabular weights](https://huggingface.co/nvidia/Kumo-Tabular). See the [KumoTabular API](https://nvidia.github.io/structured-data-models/api/generated/sdm.models.KumoTabular.html). Synthetic metrics are **demonstration only**.


## Install and configure

Install the package into this notebook kernel.
This local CPU example downloads public pretrained weights and writes only to a new output directory.

In [ ]:
%pip install structured-data-models pandas pyarrow scikit-learn

## Set the example parameters

Choose disjoint context, evaluation, and prospective cohorts. The outcome windows finish before the next cohort's cutoff; keep the threshold fixed before inspecting evaluation results.

In [ ]:
SEED = 1203
N_CONTEXT, N_TEST, N_SCORE = 2500, 1000, 500
CONTEXT_TIME = "2026-01-01"
TEST_TIME = "2026-02-15"
SCORE_TIME = "2026-04-01"
OBSERVATION_DAYS = 30
HORIZON_DAYS = 30
THRESHOLD = 0.5

MODEL_SIZE = "small"
DEVICE = "cpu"
CPU_THREADS = 2
SCRATCH_PARENT = None  # Use the operating system's temporary directory.
CLEANUP = False

NUM_ESTIMATORS = 1  # The gradient explanation below requires one estimator.
EXPLAIN_ROWS = 2


Import the direct model and analysis libraries used by the following cells.


In [ ]:
from pathlib import Path
from importlib.metadata import version
import hashlib
import tempfile

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from huggingface_hub import hf_hub_download
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

import sdm
from sdm.models import KumoTabular
from sdm.explain import GradientExplainer

Initialize CPU execution and the random generator. The printed directory is where this run will save predictions and context; existing directories are left alone.

In [ ]:
torch.set_num_threads(CPU_THREADS)
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)
owned_directory = tempfile.TemporaryDirectory(
    prefix="account-closure-", dir=SCRATCH_PARENT, delete=False
)
run_dir = Path(owned_directory.name)
print("SDM:", version("structured-data-models"), "| Device:", DEVICE)

print("Outputs:", run_dir)

## Generate synthetic example data

Run the next cell to create account snapshots, historical aggregate features, and simulated future closure labels. The prospective cohort has unknown outcomes. You can skim the generation code; the model workflow starts at **Separate context, evaluation and scoring rows**.

In [ ]:
cohort_sizes = (N_CONTEXT, N_TEST, N_SCORE)
snapshot_times = pd.to_datetime(
    np.repeat([CONTEXT_TIME, TEST_TIME, SCORE_TIME], cohort_sizes)
)
cohort = np.repeat(["context", "test", "score"], cohort_sizes)
account_count = sum(cohort_sizes)
accounts = pd.DataFrame({
    "account_id": [f"synthetic_account_{i:05d}" for i in range(account_count)],
    "cohort": cohort,
    "snapshot_time": snapshot_times,
    "observation_start": snapshot_times - pd.Timedelta(days=OBSERVATION_DAYS),
    "last_observation_at": snapshot_times - pd.to_timedelta(rng.integers(1, 24, account_count), unit="h"),
})


accounts["product_type"] = rng.choice(["checking", "savings"], account_count)
accounts["segment"] = rng.choice(["standard", "plus", "student"], account_count)
accounts["region"] = rng.choice(["region_a", "region_b", "region_c"], account_count)
accounts["balance"] = rng.lognormal(7.4, 1.0, account_count).round(2)
accounts["inflow_30d"] = rng.lognormal(7.6, 0.7, account_count).round(2)
accounts["outflow_30d"] = rng.lognormal(7.4, 0.8, account_count).round(2)
accounts["transaction_count_30d"] = rng.poisson(20, account_count)
accounts["digital_login_count_30d"] = rng.poisson(5, account_count)
accounts["support_contacts_30d"] = rng.poisson(1.2, account_count)
accounts["monthly_fee"] = rng.choice([0.0, 5.0, 12.0, 20.0], account_count)
accounts["tenure_days"] = rng.integers(90, 3000, account_count)
display(accounts.head(3))


assert accounts.account_id.notna().all() and accounts.account_id.is_unique
assert (accounts.observation_start < accounts.last_observation_at).all()
assert (accounts.last_observation_at < accounts.snapshot_time).all()
assert accounts.tenure_days.gt(OBSERVATION_DAYS).all()
feature_columns = [
    "product_type", "segment", "region", "balance", "inflow_30d", "outflow_30d",
    "transaction_count_30d", "digital_login_count_30d", "support_contacts_30d",
    "monthly_fee", "tenure_days",
]
features = accounts[feature_columns].copy()
assert not features.isna().any().any()
assert np.isfinite(features.select_dtypes(include="number").to_numpy()).all()
print(f"Prepared {len(features)} accounts with {len(feature_columns)} features each.")


known_outcomes = accounts.cohort.ne("score")
horizon_end = accounts.snapshot_time + pd.Timedelta(days=HORIZON_DAYS)
accounts["observation_end"] = horizon_end.where(known_outcomes)
accounts["closed_at"] = pd.NaT
known_accounts = accounts.loc[known_outcomes]
log_odds = (-1.3 + 0.07 * known_accounts.monthly_fee
            - 0.25 * known_accounts.digital_login_count_30d
            + 0.55 * known_accounts.support_contacts_30d
            + rng.normal(0, 0.6, known_outcomes.sum()))
closes = rng.binomial(1, 1 / (1 + np.exp(-log_odds))).astype(bool)
closure_offsets = pd.to_timedelta(rng.integers(1, HORIZON_DAYS + 1, known_outcomes.sum()), unit="D")
closure_dates = (known_accounts.snapshot_time + closure_offsets).where(closes)
accounts.loc[known_outcomes, "closed_at"] = closure_dates
mature = accounts.observation_end.notna() & (accounts.observation_end >= horizon_end)
closed_in_horizon = accounts.closed_at.gt(accounts.snapshot_time) & accounts.closed_at.le(horizon_end)
labels = closed_in_horizon.astype(float).where(mature)
accounts["closure_in_horizon"] = labels
assert labels[cohort == "score"].isna().all()
assert set(labels[cohort == "context"]) == set(labels[cohort == "test"]) == {0, 1}
print(f"Known outcomes: {labels.notna().sum()}; prospective outcomes: {labels.isna().sum()}.")

accounts.to_csv(run_dir / "accounts.csv", index=False)


## Separate context, evaluation and scoring rows

Supply only the earlier context population's labels to the model. Reserve the later held-out accounts and their labels for evaluation; do not tune model settings or thresholds on them. Prospective accounts are scored without labels.

The context outcome window ends before the evaluation cutoff, and the evaluation outcome window ends before prospective scoring. The feature allowlist excludes all IDs, timestamps, population markers and future outcomes.


In [ ]:
context_mask = cohort == "context"
test_mask = cohort == "test"
score_mask = cohort == "score"
assert pd.Timestamp(CONTEXT_TIME) + pd.Timedelta(days=HORIZON_DAYS) < pd.Timestamp(TEST_TIME)
assert pd.Timestamp(TEST_TIME) + pd.Timedelta(days=HORIZON_DAYS) < pd.Timestamp(SCORE_TIME)

context_ids = set(accounts.loc[context_mask, "account_id"])
assert context_ids.isdisjoint(accounts.loc[test_mask | score_mask, "account_id"])
assert set(accounts.loc[test_mask, "account_id"]).isdisjoint(accounts.loc[score_mask, "account_id"])

context_df = features.loc[context_mask].reset_index(drop=True)
context_df["closure_in_horizon"] = labels[context_mask].astype(int).to_numpy()
test_df = features.loc[test_mask].reset_index(drop=True)
score_df = features.loc[score_mask].reset_index(drop=True)
y_test = labels[test_mask].astype(int).to_numpy()
heldout_ids = accounts.loc[test_mask, "account_id"].reset_index(drop=True)
requested_ids = accounts.loc[score_mask, "account_id"].reset_index(drop=True)
assert "closure_in_horizon" not in test_df and "closure_in_horizon" not in score_df
print(f"Context: {len(context_df)} | Held-out: {len(test_df)} | Prospective: {len(score_df)}")


Convert the frames to SDM tensors. Infer feature types from context only, and explicitly mark the binary outcome as categorical. Evaluation and prospective tensors contain features only.

In [ ]:
context_table = sdm.TableTensor.from_pandas(
    context_df,
    stypes=sdm.infer_stypes(context_df, overrides={"closure_in_horizon": "categorical"}),
    device=DEVICE,
)
x_context = context_table.drop_columns("closure_in_horizon")
y_context = context_table[:, "closure_in_horizon"]
feature_stypes = sdm.infer_stypes(context_df[feature_columns])
x_test = sdm.TableTensor.from_pandas(test_df, stypes=feature_stypes, device=DEVICE)
x_score = sdm.TableTensor.from_pandas(score_df, stypes=feature_stypes, device=DEVICE)

## Load the small pretrained classifier

Use the small model to keep this CPU example quick. These fixed settings demonstrate the workflow; they are not selected using held-out results. The pretrained weights use the [OpenMDW license](https://huggingface.co/nvidia/Kumo-Tabular); SDM source uses Apache-2.0.

Download the model's v1.0.1 checkpoint and verify its checksum before loading. Only pretrained weights are downloaded; the example dataset is generated locally.


In [ ]:
assert MODEL_SIZE == "small", "This example verifies the small classifier checkpoint."
checkpoint_path = Path(hf_hub_download(
    "nvidia/Kumo-Tabular", "small/classifier.pt", revision="v1.0.1"
))
expected_checkpoint_hash = "1ff91f484e19021aaf7d9eff6cc07e6d95a60b4aaad473e83e2ab2c3bfd2b617"
with checkpoint_path.open("rb") as checkpoint_file:
    assert hashlib.file_digest(checkpoint_file, "sha256").hexdigest() == expected_checkpoint_hash

model = KumoTabular(task="classification", size=MODEL_SIZE, device=DEVICE)
print("Ready: small KumoTabular on CPU.")

## Fit the context and evaluate held-out accounts

`fit` prepares the known examples as context; it does not update the pretrained weights through gradient training. Use one estimator because the account-sensitivity explainer below requires it.


In [ ]:
model.fit(
    x=x_context, y=y_context, num_estimators=NUM_ESTIMATORS,
    generator=torch.Generator().manual_seed(SEED),
)

Read the positive class by its name, `"1"`, rather than by column position. The helper checks the number of rows and that each row contains finite probabilities summing to one.

In [ ]:
def positive_probability(prediction, expected_rows):
    class_names = list(prediction.columns[sdm.Stype.numerical])
    probabilities = prediction.numerical.detach().cpu().numpy()
    assert probabilities.shape == (expected_rows, len(class_names)) and "1" in class_names
    assert np.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all()
    assert np.allclose(probabilities.sum(axis=1), 1, atol=1e-5)
    return probabilities[:, class_names.index("1")]

test_prediction = model.predict(x_test)
p_test = positive_probability(test_prediction, len(test_df))

Compare predictions with a constant baseline: the fraction of context accounts that closed. AUROC measures ranking; average precision depends on prevalence; Brier score measures probability error (lower is better). These held-out metrics are **demonstration only** and do not establish real-world performance.

In [ ]:
baseline = np.full(len(y_test), context_df.closure_in_horizon.mean())
metrics = pd.DataFrame({
    "KumoTabular": [roc_auc_score(y_test, p_test), average_precision_score(y_test, p_test),
                    brier_score_loss(y_test, p_test)],
    "Constant baseline": [roc_auc_score(y_test, baseline), average_precision_score(y_test, baseline),
                          brier_score_loss(y_test, baseline)],
}, index=["AUROC", "Average precision", "Brier score"])
assert np.isfinite(metrics.to_numpy()).all()
display(metrics.round(3))

## Batch predictions

Reuse the fitted context to score the prospective cohort. The result has one account ID and one closure probability per requested account. `above_threshold` is an illustrative review flag. Save the table for inspection; it is not an automated account decision.

In [ ]:
score_prediction = model.predict(x_score)

### Inspect and save the prospective scores

Read the positive class by name, verify one probability per account in input order, and inspect the illustrative review flag before saving the full batch.

In [ ]:
p_score = positive_probability(score_prediction, len(score_df))
predictions = pd.DataFrame({
    "account_id": requested_ids,
    "closure_probability": p_score,
    "above_threshold": p_score >= THRESHOLD,
})
assert predictions.account_id.is_unique and predictions.account_id.tolist() == requested_ids.tolist()
predictions.to_parquet(run_dir / "predictions.parquet", index=False)
display(predictions.head(5))
print(f"Saved {len(predictions)} account predictions.")

## Save context and reproduce the scores

Save the labeled context and prospective inputs, keeping account IDs for bookkeeping. Rebuild a fresh pretrained model from those files using this notebook's configured model and estimator settings.


In [ ]:
context_df.to_parquet(run_dir / "context.parquet", index=False)
saved_scoring_rows = score_df.copy()
saved_scoring_rows.insert(0, "account_id", requested_ids)
saved_scoring_rows.to_parquet(run_dir / "scoring_rows.parquet", index=False)
print("Saved predictions, context and scoring rows in", run_dir)

Reload those files and rebuild the same context in a new model instance. This demonstrates reconstruction in the current Python process, not a serialized model bundle or a fresh-process test. The expected result is the same account IDs and probabilities.

In [ ]:
restored_context = pd.read_parquet(run_dir / "context.parquet")
restored_scoring_rows = pd.read_parquet(run_dir / "scoring_rows.parquet")
restored_ids = restored_scoring_rows.pop("account_id")
restored_table = sdm.TableTensor.from_pandas(
    restored_context,
    stypes=sdm.infer_stypes(restored_context, overrides={"closure_in_horizon": "categorical"}),
    device=DEVICE,
)
restored_query = sdm.TableTensor.from_pandas(
    restored_scoring_rows, stypes=sdm.infer_stypes(restored_context[feature_columns]), device=DEVICE
)
restored_model = KumoTabular(task="classification", size=MODEL_SIZE, device=DEVICE)
restored_model.fit(
    x=restored_table.drop_columns("closure_in_horizon"), y=restored_table[:, "closure_in_horizon"],
    num_estimators=NUM_ESTIMATORS, generator=torch.Generator().manual_seed(SEED),
)
restored_probabilities = positive_probability(restored_model.predict(restored_query), len(restored_ids))
reloaded_predictions = pd.DataFrame({"account_id": restored_ids, "closure_probability": restored_probabilities})
assert reloaded_predictions.account_id.tolist() == requested_ids.tolist()
reconstruction_matches = np.allclose(restored_probabilities, p_score, rtol=1e-5, atol=1e-6)
assert reconstruction_matches
print("Rebuilt model context reproduces all prospective predictions.")

## Inspect account sensitivity

[GradientExplainer](https://nvidia.github.io/structured-data-models/api/generated/sdm.explain.GradientExplainer.html) measures how output probabilities change with the model's **preprocessed numerical inputs**. Categories may be encoded and columns transformed or reordered, so use the returned column names. A large absolute gradient indicates local model sensitivity; it is not a causal effect or a change per original currency unit.

Explain the first two prospective accounts and summarize their positive-class gradients. These explanations describe only the selected accounts, not all model behavior.

In [ ]:
attributions, related_attributions = GradientExplainer().explain(model, x_score[:EXPLAIN_ROWS])
gradients = attributions.numerical.detach().cpu().numpy()
assert related_attributions is None and np.isfinite(gradients).all()
assert gradients.shape[-2] == EXPLAIN_ROWS
positive_class_index = list(score_prediction.columns[sdm.Stype.numerical]).index("1")
positive_gradients = gradients[positive_class_index].reshape(-1, gradients.shape[-1])
explanation_summary = pd.DataFrame({
    "preprocessed_column": list(attributions.columns[sdm.Stype.numerical]),
    "mean_absolute_gradient": np.abs(positive_gradients).mean(axis=0),
}).sort_values("mean_absolute_gradient", ascending=False)
print("Explained accounts:", requested_ids.iloc[:EXPLAIN_ROWS].tolist())
display(explanation_summary.head(5))

## Cleanup

Release context from both model instances. Set `CLEANUP=True` to remove only this run's output directory; the default retains its files and the shared pretrained-weight cache.

In [ ]:
model.clear()
restored_model.clear()
if CLEANUP:
    owned_directory.cleanup()
    print("Removed this run's output directory.")
else:
    print("Saved files remain in", run_dir)